# BD-KDD Kidney Disease — Random Forest + DODA Feature Selection

## Experimental objective

Evaluate conventional **Random Forest feature selection** against **Random Forest + DODA clinical-priority re-ranking** on the BD-KDD kidney-disease dataset.

### Important design rule

Random Forest ranks **all 24 predictors**. DODA also receives **all 24 predictor scores** and performs the final Top-K selection after clinical-priority fusion.

This is essential: if Random Forest selected Top-K before DODA, DODA could not re-prioritize features outside that preliminary Top-K.

### Dataset

- 988 records
- 24 predictors
- Binary target: `Class`
- `Class = 0`: healthy
- `Class = 1`: kidney disease
- 0 missing values
- 80/20 stratified fixed split for the initial experiment
- 5×5 repeated stratified CV for robustness


In [1]:
# =============================================================================
# STEP 1: LOAD PROCESSED DATASET
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../../data/processed/BD-KDD_cleaned.csv")

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}"
    )

df = pd.read_csv(DATA_PATH)

print("=" * 70)
print("PROCESSED BD-KDD DATASET")
print("=" * 70)
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

display(df.head())


PROCESSED BD-KDD DATASET
Rows    : 988
Columns : 25


,Age,Bp,Sg,Al,Su,Rbc,Pc,Pcc,Ba,Bgr,...,Pcv,Wbcc,Rbcc,Htn,Dm,Cad,Appet,Pe,Ane,Class
0,58,147,1.025,1,3,0,1,0,0,177,...,43,12342,4.3,1,0,0,1,1,0,0
1,71,142,1.025,2,3,1,0,1,0,247,...,47,7249,3.7,0,1,1,0,1,1,1
2,48,179,1.005,0,4,1,1,0,1,367,...,37,4111,3.7,0,0,0,0,0,1,0
3,34,70,1.005,1,3,1,1,0,0,215,...,44,4682,4.0,1,0,1,1,0,1,1
4,62,120,1.020,4,0,1,0,1,0,143,...,54,5161,4.7,0,0,1,0,1,0,1


In [2]:
# =============================================================================
# STEP 2: FEATURE AND TARGET SEPARATION
# =============================================================================

TARGET_COLUMN = "Class"

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print("=" * 70)
print("FEATURE / TARGET SEPARATION")
print("=" * 70)
print(f"Predictors (X) : {X.shape[1]}")
print(f"Records        : {X.shape[0]}")
print(f"Target (y)     : {TARGET_COLUMN}")
print(f"Target values  : {sorted(y.unique().tolist())}")

assert X.shape[1] == 24, "Expected 24 predictors after removing Class."
assert set(y.unique()) == {0, 1}

display(X.head())
display(y.head())


FEATURE / TARGET SEPARATION
Predictors (X) : 24
Records        : 988
Target (y)     : Class
Target values  : [0, 1]


,Age,Bp,Sg,Al,Su,Rbc,Pc,Pcc,Ba,Bgr,...,Hemo,Pcv,Wbcc,Rbcc,Htn,Dm,Cad,Appet,Pe,Ane
0,58,147,1.025,1,3,0,1,0,0,177,...,9.3,43,12342,4.3,1,0,0,1,1,0
1,71,142,1.025,2,3,1,0,1,0,247,...,16.9,47,7249,3.7,0,1,1,0,1,1
2,48,179,1.005,0,4,1,1,0,1,367,...,16.6,37,4111,3.7,0,0,0,0,0,1
3,34,70,1.005,1,3,1,1,0,0,215,...,15.4,44,4682,4.0,1,0,1,1,0,1
4,62,120,1.020,4,0,1,0,1,0,143,...,13.9,54,5161,4.7,0,0,1,0,1,0


0    0
1    1
2    0
3    1
4    1
Name: Class, dtype: int64

In [3]:
# =============================================================================
# STEP 3: TARGET DISTRIBUTION
# =============================================================================

target_distribution = pd.DataFrame({
    "Count": y.value_counts().sort_index(),
    "Percentage": (y.value_counts(normalize=True).sort_index() * 100).round(2)
})

display(target_distribution)


,Count,Percentage
Class,,
0,481,48.68
1,507,51.32


In [4]:
# =============================================================================
# STEP 4: DATA TYPE AND MISSING-VALUE CHECK
# =============================================================================

print("=" * 70)
print("DATA TYPES")
print("=" * 70)
display(X.dtypes.to_frame("Data Type"))

missing = pd.DataFrame({
    "Missing Count": X.isna().sum(),
    "Missing Percentage (%)": (X.isna().mean() * 100).round(2)
})

print("=" * 70)
print("MISSING VALUES")
print("=" * 70)
display(missing)

assert X.isna().sum().sum() == 0, "Unexpected missing values detected."

print("No missing values detected. No imputation will be performed.")


DATA TYPES


,Data Type
Age,int64
Bp,int64
Sg,float64
Al,int64
Su,int64
Rbc,int64
Pc,int64
Pcc,int64
Ba,int64
Bgr,int64


MISSING VALUES


,Missing Count,Missing Percentage (%)
Age,0,0.0
Bp,0,0.0
Sg,0,0.0
Al,0,0.0
Su,0,0.0
Rbc,0,0.0
Pc,0,0.0
Pcc,0,0.0
Ba,0,0.0
Bgr,0,0.0


No missing values detected. No imputation will be performed.


## Preprocessing decision

The EDA established that BD-KDD contains **zero missing values**. Therefore, no imputation step is included in this notebook.

Feature selection is fitted only on training data/folds. Model scaling is applied only where needed and is fitted only on training data/folds.


In [5]:
# =============================================================================
# STEP 5: FIXED 80/20 STRATIFIED TRAIN-TEST SPLIT
# =============================================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("=" * 70)
print("TRAIN-TEST SPLIT")
print("=" * 70)
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

print("\nTraining class distribution:")
display((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting class distribution:")
display((y_test.value_counts(normalize=True) * 100).round(2))


TRAIN-TEST SPLIT
X_train : (790, 24)
X_test  : (198, 24)
y_train : (790,)
y_test  : (198,)

Training class distribution:


Class
1    51.27
0    48.73
Name: proportion, dtype: float64


Testing class distribution:


Class
1    51.52
0    48.48
Name: proportion, dtype: float64

In [6]:
# =============================================================================
# STEP 6: FEATURE-SCALE CHECK
# =============================================================================
# RF is applied to the training data after fold-specific preprocessing.
# Scaling is fitted only on the corresponding training partition.
#
# Logistic Regression is scaled again after feature selection.
# Random Forest and XGBoost use the selected features without downstream scaling.

print("RF will be fitted using training data only.")
print("Logistic Regression will be scaled after feature selection.")
print("Random Forest and XGBoost will use the selected features without scaling.")


RF will be fitted using training data only.
Logistic Regression will be scaled after feature selection.
Random Forest and XGBoost will use the selected features without scaling.


# 1. Random Forest Baseline

Random Forest is used as the statistical feature-selection baseline.

**Important:** Random Forest is fitted on all 24 predictors first. The complete Random Forest ranking is generated before Top-K is applied.


In [7]:
# =============================================================================
# STEP 7: RANDOM FOREST FEATURE IMPORTANCE RANKING FOR ALL FEATURES
# =============================================================================

from sklearn.ensemble import RandomForestClassifier

rf_selector_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

# Feature selection is learned from the training split only.
rf_selector_model.fit(X_train, y_train)

rf_importances = pd.Series(
    rf_selector_model.feature_importances_,
    index=X_train.columns,
    name="RF_Importance"
).sort_values(ascending=False)

rf_scores = rf_importances.reset_index()
rf_scores.columns = ["Feature", "RF_Importance"]

rf_scores["RF_Rank"] = np.arange(
    1,
    len(rf_scores) + 1
)

display(rf_scores)


,Feature,RF_Importance,RF_Rank
0,Bp,0.080438,1
1,Bu,0.079633,2
2,Sc,0.078954,3
3,Wbcc,0.075748,4
4,Hemo,0.074285,5
5,Bgr,0.073153,6
6,Age,0.071706,7
7,Pot,0.070899,8
8,Pcv,0.060499,9
9,Rbcc,0.060186,10


In [8]:
# =============================================================================
# STEP 8: TOP-K RANDOM FOREST FEATURE SETS
# =============================================================================

TOP_K_VALUES = [5, 10, 15, 20]

print(f"Total predictors: {X.shape[1]}")
print(f"Top-K values: {TOP_K_VALUES}")

rf_results = {}

for top_k in TOP_K_VALUES:

    selected = (
        rf_scores
        .head(top_k)["Feature"]
        .tolist()
    )

    rf_results[top_k] = {
        "features": selected,
        "X_train": X_train[selected].copy(),
        "X_test": X_test[selected].copy()
    }

    print("=" * 70)
    print(f"RF TOP-{top_k}")
    print("=" * 70)
    print(selected)


Total predictors: 24
Top-K values: [5, 10, 15, 20]
RF TOP-5
['Bp', 'Bu', 'Sc', 'Wbcc', 'Hemo']
RF TOP-10
['Bp', 'Bu', 'Sc', 'Wbcc', 'Hemo', 'Bgr', 'Age', 'Pot', 'Pcv', 'Rbcc']
RF TOP-15
['Bp', 'Bu', 'Sc', 'Wbcc', 'Hemo', 'Bgr', 'Age', 'Pot', 'Pcv', 'Rbcc', 'Sod', 'Su', 'Al', 'Sg', 'Appet']
RF TOP-20
['Bp', 'Bu', 'Sc', 'Wbcc', 'Hemo', 'Bgr', 'Age', 'Pot', 'Pcv', 'Rbcc', 'Sod', 'Su', 'Al', 'Sg', 'Appet', 'Ba', 'Pc', 'Pcc', 'Htn', 'Pe']


In [9]:
# =============================================================================
# STEP 9: DOWNSTREAM MODELS
# =============================================================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

print(list(models.keys()))


['Logistic Regression', 'Random Forest', 'XGBoost']


In [10]:
# =============================================================================
# STEP 10: RANDOM FOREST BASELINE MODEL EVALUATION
# =============================================================================

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

fixed_rf_rows = []

for top_k in TOP_K_VALUES:

    Xtr_selected = rf_results[top_k]["X_train"]
    Xte_selected = rf_results[top_k]["X_test"]
    selected_features = rf_results[top_k]["features"]

    for model_name, model_template in models.items():

        model = model_template.__class__(
            **model_template.get_params()
        )

        Xtr = Xtr_selected
        Xte = Xte_selected

        if model_name == "Logistic Regression":
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(Xtr)
            Xte = scaler.transform(Xte)

        model.fit(Xtr, y_train)

        y_pred = model.predict(Xte)
        y_prob = model.predict_proba(Xte)[:, 1]

        fixed_rf_rows.append({
            "Method": "RF",
            "Top_K": top_k,
            "Model": model_name,
            "Selected_Features": ", ".join(selected_features),
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0),
            "ROC_AUC": roc_auc_score(y_test, y_prob)
        })

fixed_rf_results_df = pd.DataFrame(fixed_rf_rows)
display(fixed_rf_results_df.round(4))


,Method,Top_K,Model,Selected_Features,Accuracy,Precision,Recall,F1,ROC_AUC
0,RF,5,Logistic Regression,"Bp, Bu, Sc, Wbcc, Hemo",0.5202,0.5304,0.5980,0.5622,0.5382
1,RF,5,Random Forest,"Bp, Bu, Sc, Wbcc, Hemo",0.4747,0.4886,0.4216,0.4526,0.4995
2,RF,5,XGBoost,"Bp, Bu, Sc, Wbcc, Hemo",0.4949,0.5102,0.4902,0.5000,0.5008
3,RF,10,Logistic Regression,"Bp, Bu, Sc, Wbcc, Hemo, Bgr, Age, Pot, Pcv, Rbcc",0.5707,0.5766,0.6275,0.6009,0.5816
4,RF,10,Random Forest,"Bp, Bu, Sc, Wbcc, Hemo, Bgr, Age, Pot, Pcv, Rbcc",0.5505,0.5670,0.5392,0.5528,0.5857
5,RF,10,XGBoost,"Bp, Bu, Sc, Wbcc, Hemo, Bgr, Age, Pot, Pcv, Rbcc",0.4798,0.4949,0.4804,0.4876,0.4987
6,RF,15,Logistic Regression,"Bp, Bu, Sc, Wbcc, Hemo, Bgr, Age, Pot, Pcv, Rb...",0.5909,0.5946,0.6471,0.6197,0.5890
7,RF,15,Random Forest,"Bp, Bu, Sc, Wbcc, Hemo, Bgr, Age, Pot, Pcv, Rb...",0.5505,0.5644,0.5588,0.5616,0.5794
8,RF,15,XGBoost,"Bp, Bu, Sc, Wbcc, Hemo, Bgr, Age, Pot, Pcv, Rb...",0.5101,0.5243,0.5294,0.5268,0.5036
9,RF,20,Logistic Regression,"Bp, Bu, Sc, Wbcc, Hemo, Bgr, Age, Pot, Pcv, Rb...",0.5606,0.5701,0.5980,0.5837,0.5908


In [11]:
# =============================================================================
# STEP 11: SAVE RF BASELINE
# =============================================================================

baseline_dir = Path("../../../results/kidney_disease/baseline")
baseline_dir.mkdir(parents=True, exist_ok=True)

baseline_path = baseline_dir / "rf_baseline_results.csv"
fixed_rf_results_df.to_csv(baseline_path, index=False)

print(f"Saved: {baseline_path.resolve()}")


Saved: C:\Users\johnm\msc_research\results\kidney_disease\baseline\rf_baseline_results.csv


# 2. DODA Clinical Re-ranking

### Correct experimental order

```text
24 original predictors
        ↓
Random Forest ranking for all 24
        ↓
statistical ranking
        +
pre-specified clinical weights
        ↓
DODA RankFusion
        ↓
final Top-K
```

DODA must **not** receive an already-truncated Random Forest Top-K set. Otherwise, clinically prioritized features outside that preliminary Top-K would be impossible to recover.


In [12]:
# =============================================================================
# STEP 12: INSTALL / IMPORT DODA
# =============================================================================

# Run the following installation once if DODA is not installed:
# %pip install --no-cache-dir git+https://github.com/anandha-3679/DODA.git

import doda

from doda import DODASelector
from doda.adapters import SklearnAdapter
from doda.knowledge.providers import JSONProvider
from doda.fusion import RankFusion

print("DODA:", doda.__file__)

print("Provider:", JSONProvider)
print("Fusion:", RankFusion)

DODA: c:\Users\johnm\msc_research\.venv\Lib\site-packages\doda\__init__.py
Provider: <class 'doda.knowledge.providers.json_provider.JSONProvider'>
Fusion: <class 'doda.fusion.rank.RankFusion'>


In [13]:
# =============================================================================
# STEP 13: LOCATE AND VALIDATE CLINICAL WEIGHTS
# =============================================================================

from pathlib import Path
from doda.knowledge.providers import JSONProvider

WEIGHTS_FILENAME = "BD_KDD_clinical_weights.json"

search_roots = [Path.cwd(), *Path.cwd().parents]
matches = []

for root in search_roots:
    try:
        matches.extend(root.rglob(WEIGHTS_FILENAME))
    except (PermissionError, OSError):
        continue

# Remove duplicates while preserving order
matches = list(dict.fromkeys(matches))

if not matches:
    raise FileNotFoundError(
        f"{WEIGHTS_FILENAME} was not found. "
        "Place it in the repository and rerun this cell."
    )

WEIGHTS_PATH = matches[0]

print(f"Using clinical weights: {WEIGHTS_PATH.resolve()}")

# Use the standard DODA JSON provider
provider = JSONProvider(str(WEIGHTS_PATH))

# Validate coverage using the flat JSON directly
import json

with open(WEIGHTS_PATH, "r", encoding="utf-8") as f:
    clinical_weights = json.load(f)

missing_weights = sorted(
    set(X.columns) - set(clinical_weights)
)

extra_weights = sorted(
    set(clinical_weights) - set(X.columns)
)

print(f"Clinical weights: {len(clinical_weights)}")
print(f"Predictors:       {len(X.columns)}")
print(f"Missing weights:  {missing_weights}")
print(f"Extra weights:    {extra_weights}")

assert not missing_weights
assert not extra_weights

Using clinical weights: C:\Users\johnm\msc_research\config\clinical_weights\BD_KDD_clinical_weights.json
Clinical weights: 24
Predictors:       24
Missing weights:  []
Extra weights:    []


In [14]:
# =============================================================================
# STEP 14: DODA — RF SCORE ALL FEATURES, THEN APPLY TOP-K
# =============================================================================

class RFAdapter:
    """
    Adapter that fits a Random Forest on all available features and exposes
    the model's feature_importances_ as the statistical/operator scores.
    """

    def __init__(self):
        self.importance = {}

    def fit(self, X, y):
        X_local = pd.DataFrame(X).copy()
        X_local.columns = list(X.columns)
        y_local = pd.Series(y).reset_index(drop=True)

        rf_model = RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )

        rf_model.fit(X_local, y_local)

        self.importance = dict(
            zip(
                X_local.columns,
                rf_model.feature_importances_
            )
        )

        return self

    def get_importance(self):
        return self.importance

    def get_name(self):
        return "RF"


rank_doda_results = {}

for top_k in TOP_K_VALUES:

    rf_operator = RFAdapter()

    doda_selector = DODASelector(
        operators=[rf_operator],
        provider=provider,
        fusion=RankFusion(),
        top_k=top_k
    )

    # Fit on the complete training split only.
    doda_selector.fit(
        X_train,
        y_train
    )

    selected_features = list(
        doda_selector.get_selected_features()
    )

    rank_doda_results[top_k] = {
        "selector": doda_selector,
        "features": selected_features,
        "X_train": X_train[selected_features].copy(),
        "X_test": X_test[selected_features].copy()
    }

    print("=" * 70)
    print(f"DODA TOP-{top_k}")
    print("=" * 70)
    print(selected_features)
    print(f"Number selected: {len(selected_features)}")

    assert len(selected_features) == top_k


Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x00000217F60EC980>
DODA Pipeline Started
Running: RF
Operator Scores:
{'RF_1': {'Age': np.float64(0.07170591339195666), 'Bp': np.float64(0.0804383772721103), 'Sg': np.float64(0.030370463971245803), 'Al': np.float64(0.031706117843900726), 'Su': np.float64(0.032493259769639064), 'Rbc': np.float64(0.01185462691148671), 'Pc': np.float64(0.012605871494316051), 'Pcc': np.float64(0.012311993966076224), 'Ba': np.float64(0.012752702851229765), 'Bgr': np.float64(0.07315298360456213), 'Bu': np.float64(0.07963321250528797), 'Sc': np.float64(0.0789544089989916), 'Sod': np.float64(0.05851818728230143), 'Pot': np.float64(0.07089850412451618), 'Hemo': np.float64(0.07428464683878057), 'Pcv': np.float64(0.060499381908626186), 'Wbcc': np.float64(0.0757483012173945), 'Rbcc': np.float64(0.06018628947040401), 'Htn': np.float64(0.01213220785527051), 'Dm': np.float64(0.011750336462267412), 'Cad': np.float64(0.011

In [15]:
# =============================================================================
# STEP 15: INSPECT DODA RANK CHANGES
# =============================================================================

INSPECTION_TOP_K = 10

selector = rank_doda_results[INSPECTION_TOP_K]["selector"]

print("=" * 70)
print(f"DODA RANK COMPARISON — TOP-{INSPECTION_TOP_K}")
print("=" * 70)

rank_comparison = selector.compare_scores()

display(rank_comparison)

print("\nSelected features:")
print(rank_doda_results[INSPECTION_TOP_K]["features"])


DODA RANK COMPARISON — TOP-10

MATHEMATICAL vs CLINICAL vs RANKFUSION

Fusion Method: RankFusion

Top Features After Fusion:
  Feature  Math Rank  Clinical Rank  Final Rank  Rank Change
0      Sc          3              2           1            2
1      Bu          2              3           2            0
2      Bp          1              6           3           -2
3    Hemo          5              8           4            1
4     Bgr          6              9           5            1
5      Al         13              1           6            7
6    Wbcc          4             23           7           -3
7     Pcv          9             10           8            1
8     Age          7             15           9           -2
9     Pot          8             13          10           -2


,Feature,Math Rank,Normalized Math Score,Clinical Rank,Clinical Weight,Final Rank,Final Score,Rank Change
0,Sc,3,0.9816,2,1.00,1,0.0323,2
1,Bu,2,0.9900,3,0.90,2,0.0323,0
2,Bp,1,1.0000,6,0.80,3,0.0320,-2
3,Hemo,5,0.9235,8,0.75,4,0.0308,1
4,Bgr,6,0.9094,9,0.70,5,0.0303,1
5,Al,13,0.3942,1,1.00,6,0.0301,7
6,Wbcc,4,0.9417,23,0.40,7,0.0297,-3
7,Pcv,9,0.7521,10,0.70,8,0.0296,1
8,Age,7,0.8914,15,0.55,9,0.0296,-2
9,Pot,8,0.8814,13,0.65,10,0.0296,-2



Selected features:
['Sc', 'Bu', 'Bp', 'Hemo', 'Bgr', 'Al', 'Wbcc', 'Pcv', 'Age', 'Pot']


In [16]:
# =============================================================================
# STEP 15B: EXPLICIT RF vs DODA RANK-SHIFT TABLE
# =============================================================================
# This table separates:
#   - statistical RF rank
#   - DODA final rank
#   - clinical weight
#   - rank movement
#
# Positive Rank_Change means the feature moved upward.
# Example: RF rank 10 -> DODA rank 4 = +6
#
# IMPORTANT:
# The selector is explicitly retrieved from rank_doda_results
# for the selected Top-K value. This avoids relying on a
# previously defined 'selector' variable.

# -------------------------------------------------------------------------
# Select which DODA Top-K experiment to inspect
# -------------------------------------------------------------------------

k = 10

if k not in rank_doda_results:
    raise KeyError(
        f"Top-K={k} was not found in rank_doda_results. "
        f"Available values: {list(rank_doda_results.keys())}"
    )

selector = rank_doda_results[k]["selector"]


# -------------------------------------------------------------------------
# RF / mathematical ranking
# -------------------------------------------------------------------------

math_rank = pd.DataFrame(
    selector.math_scores_.items(),
    columns=[
        "Feature",
        "RF_Importance"
    ]
)

math_rank["RF_Rank"] = (
    math_rank["RF_Importance"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


# -------------------------------------------------------------------------
# DODA final ranking
# -------------------------------------------------------------------------

final_rank = pd.DataFrame(
    selector.final_scores_.items(),
    columns=[
        "Feature",
        "DODA_Final_Score"
    ]
)

final_rank["DODA_Rank"] = (
    final_rank["DODA_Final_Score"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


# -------------------------------------------------------------------------
# Combine rankings
# -------------------------------------------------------------------------

rank_shift_df = (
    math_rank[
        [
            "Feature",
            "RF_Importance",
            "RF_Rank"
        ]
    ]
    .merge(
        final_rank[
            [
                "Feature",
                "DODA_Final_Score",
                "DODA_Rank"
            ]
        ],
        on="Feature",
        how="inner"
    )
)


# -------------------------------------------------------------------------
# Add clinical weights
# -------------------------------------------------------------------------

rank_shift_df["Clinical_Weight"] = (
    rank_shift_df["Feature"]
    .map(selector.clinical_weights_)
)


# -------------------------------------------------------------------------
# Calculate rank movement
# -------------------------------------------------------------------------
# Positive value  = moved upward
# Negative value  = moved downward
# Zero             = no rank movement

rank_shift_df["Rank_Change"] = (
    rank_shift_df["RF_Rank"]
    - rank_shift_df["DODA_Rank"]
)


# -------------------------------------------------------------------------
# Sort by DODA final rank
# -------------------------------------------------------------------------

rank_shift_df = (
    rank_shift_df
    .sort_values(
        by="DODA_Rank"
    )
    .reset_index(drop=True)
)


# -------------------------------------------------------------------------
# Display
# -------------------------------------------------------------------------

print("=" * 80)
print(f"RF vs DODA RANK-SHIFT ANALYSIS — TOP-{k}")
print("=" * 80)

display(rank_shift_df)

RF vs DODA RANK-SHIFT ANALYSIS — TOP-10


,Feature,RF_Importance,RF_Rank,DODA_Final_Score,DODA_Rank,Clinical_Weight,Rank_Change
0,Sc,0.981551,3,0.032266,1,1.00,2
1,Bu,0.989990,2,0.032258,2,0.90,0
2,Bp,1.000000,1,0.032018,3,0.80,-2
3,Hemo,0.923498,5,0.030769,4,0.75,1
4,Bgr,0.909429,6,0.030303,5,0.70,1
5,Al,0.394167,13,0.030092,6,1.00,7
6,Wbcc,0.941694,4,0.029710,7,0.40,-3
7,Pcv,0.752121,9,0.029644,8,0.70,1
8,Pot,0.881401,8,0.029631,9,0.65,-1
9,Age,0.891439,7,0.029631,9,0.55,-2


In [17]:
# =============================================================================
# STEP 16: INSPECT CLINICAL WEIGHTS
# =============================================================================

clinical_weight_table = pd.DataFrame({
    "Feature": list(selector.clinical_weights_.keys()),
    "Clinical_Weight": list(selector.clinical_weights_.values())
}).sort_values(
    "Clinical_Weight",
    ascending=False
).reset_index(drop=True)

display(clinical_weight_table)


,Feature,Clinical_Weight
0,Al,1.00
1,Sc,1.00
2,Bu,0.90
3,Sg,0.85
4,Htn,0.85
5,Dm,0.80
6,Bp,0.80
7,Hemo,0.75
8,Pcv,0.70
9,Ane,0.70


In [18]:
# =============================================================================
# STEP 17: DODA MODEL EVALUATION — FIXED SPLIT
# =============================================================================

fixed_doda_rows = []

for top_k in TOP_K_VALUES:

    Xtr_selected = rank_doda_results[top_k]["X_train"]
    Xte_selected = rank_doda_results[top_k]["X_test"]
    selected_features = rank_doda_results[top_k]["features"]

    for model_name, model_template in models.items():

        model = model_template.__class__(
            **model_template.get_params()
        )

        Xtr = Xtr_selected
        Xte = Xte_selected

        if model_name == "Logistic Regression":
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(Xtr)
            Xte = scaler.transform(Xte)

        model.fit(Xtr, y_train)

        y_pred = model.predict(Xte)
        y_prob = model.predict_proba(Xte)[:, 1]

        fixed_doda_rows.append({
            "Method": "DODA",
            "Top_K": top_k,
            "Model": model_name,
            "Selected_Features": ", ".join(selected_features),
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1": f1_score(y_test, y_pred, zero_division=0),
            "ROC_AUC": roc_auc_score(y_test, y_prob)
        })

fixed_doda_results_df = pd.DataFrame(fixed_doda_rows)
display(fixed_doda_results_df.round(4))


,Method,Top_K,Model,Selected_Features,Accuracy,Precision,Recall,F1,ROC_AUC
0,DODA,5,Logistic Regression,"Sc, Bu, Bp, Hemo, Bgr",0.5505,0.5565,0.6275,0.5899,0.5472
1,DODA,5,Random Forest,"Sc, Bu, Bp, Hemo, Bgr",0.5455,0.5600,0.5490,0.5545,0.5715
2,DODA,5,XGBoost,"Sc, Bu, Bp, Hemo, Bgr",0.5354,0.5510,0.5294,0.5400,0.5461
3,DODA,10,Logistic Regression,"Sc, Bu, Bp, Hemo, Bgr, Al, Wbcc, Pcv, Age, Pot",0.5404,0.5487,0.6078,0.5767,0.5661
4,DODA,10,Random Forest,"Sc, Bu, Bp, Hemo, Bgr, Al, Wbcc, Pcv, Age, Pot",0.5505,0.5657,0.5490,0.5572,0.5610
5,DODA,10,XGBoost,"Sc, Bu, Bp, Hemo, Bgr, Al, Wbcc, Pcv, Age, Pot",0.4949,0.5100,0.5000,0.5050,0.5130
6,DODA,15,Logistic Regression,"Sc, Bu, Bp, Hemo, Bgr, Al, Wbcc, Pcv, Age, Pot...",0.5859,0.5909,0.6373,0.6132,0.5846
7,DODA,15,Random Forest,"Sc, Bu, Bp, Hemo, Bgr, Al, Wbcc, Pcv, Age, Pot...",0.5455,0.5577,0.5686,0.5631,0.5668
8,DODA,15,XGBoost,"Sc, Bu, Bp, Hemo, Bgr, Al, Wbcc, Pcv, Age, Pot...",0.4899,0.5048,0.5196,0.5121,0.5025
9,DODA,20,Logistic Regression,"Sc, Bu, Bp, Hemo, Bgr, Al, Wbcc, Pcv, Age, Pot...",0.5707,0.5794,0.6078,0.5933,0.5772


In [19]:
# =============================================================================
# STEP 18: FIXED-SPLIT RF vs DODA COMPARISON
# =============================================================================

fixed_comparison_df = pd.concat(
    [fixed_rf_results_df, fixed_doda_results_df],
    ignore_index=True
)

display(fixed_comparison_df.round(4))


,Method,Top_K,Model,Selected_Features,Accuracy,Precision,Recall,F1,ROC_AUC
0,RF,5,Logistic Regression,"Bp, Bu, Sc, Wbcc, Hemo",0.5202,0.5304,0.5980,0.5622,0.5382
1,RF,5,Random Forest,"Bp, Bu, Sc, Wbcc, Hemo",0.4747,0.4886,0.4216,0.4526,0.4995
2,RF,5,XGBoost,"Bp, Bu, Sc, Wbcc, Hemo",0.4949,0.5102,0.4902,0.5000,0.5008
3,RF,10,Logistic Regression,"Bp, Bu, Sc, Wbcc, Hemo, Bgr, Age, Pot, Pcv, Rbcc",0.5707,0.5766,0.6275,0.6009,0.5816
4,RF,10,Random Forest,"Bp, Bu, Sc, Wbcc, Hemo, Bgr, Age, Pot, Pcv, Rbcc",0.5505,0.5670,0.5392,0.5528,0.5857
5,RF,10,XGBoost,"Bp, Bu, Sc, Wbcc, Hemo, Bgr, Age, Pot, Pcv, Rbcc",0.4798,0.4949,0.4804,0.4876,0.4987
6,RF,15,Logistic Regression,"Bp, Bu, Sc, Wbcc, Hemo, Bgr, Age, Pot, Pcv, Rb...",0.5909,0.5946,0.6471,0.6197,0.5890
7,RF,15,Random Forest,"Bp, Bu, Sc, Wbcc, Hemo, Bgr, Age, Pot, Pcv, Rb...",0.5505,0.5644,0.5588,0.5616,0.5794
8,RF,15,XGBoost,"Bp, Bu, Sc, Wbcc, Hemo, Bgr, Age, Pot, Pcv, Rb...",0.5101,0.5243,0.5294,0.5268,0.5036
9,RF,20,Logistic Regression,"Bp, Bu, Sc, Wbcc, Hemo, Bgr, Age, Pot, Pcv, Rb...",0.5606,0.5701,0.5980,0.5837,0.5908


In [20]:
# =============================================================================
# STEP 19: COMPARE FEATURE SETS ON THE FIXED SPLIT
# =============================================================================

fixed_feature_comparison = []

for top_k in TOP_K_VALUES:

    rf_set = set(rf_results[top_k]["features"])
    doda_set = set(rank_doda_results[top_k]["features"])

    intersection = rf_set & doda_set
    union = rf_set | doda_set

    jaccard = len(intersection) / len(union)

    fixed_feature_comparison.append({
        "Top_K": top_k,
        "RF_Features": ", ".join(rf_results[top_k]["features"]),
        "DODA_Features": ", ".join(rank_doda_results[top_k]["features"]),
        "Same_Feature_Set": rf_set == doda_set,
        "Overlap_Count": len(intersection),
        "Jaccard": jaccard,
        "Features_Changed": top_k - len(intersection)
    })

fixed_feature_comparison_df = pd.DataFrame(fixed_feature_comparison)

display(fixed_feature_comparison_df)


,Top_K,RF_Features,DODA_Features,Same_Feature_Set,Overlap_Count,Jaccard,Features_Changed
0,5,"Bp, Bu, Sc, Wbcc, Hemo","Sc, Bu, Bp, Hemo, Bgr",False,4,0.666667,1
1,10,"Bp, Bu, Sc, Wbcc, Hemo, Bgr, Age, Pot, Pcv, Rbcc","Sc, Bu, Bp, Hemo, Bgr, Al, Wbcc, Pcv, Age, Pot",False,9,0.818182,1
2,15,"Bp, Bu, Sc, Wbcc, Hemo, Bgr, Age, Pot, Pcv, Rb...","Sc, Bu, Bp, Hemo, Bgr, Al, Wbcc, Pcv, Age, Pot...",False,14,0.875000,1
3,20,"Bp, Bu, Sc, Wbcc, Hemo, Bgr, Age, Pot, Pcv, Rb...","Sc, Bu, Bp, Hemo, Bgr, Al, Wbcc, Pcv, Age, Pot...",False,18,0.818182,2


In [21]:
# =============================================================================
# STEP 20: SAVE FIXED-SPLIT DODA RESULTS
# =============================================================================

doda_dir = Path("../../../results/kidney_disease/doda")
doda_dir.mkdir(parents=True, exist_ok=True)

fixed_doda_results_df.to_csv(
    doda_dir / "rf_doda_fixed_split_results.csv",
    index=False
)

fixed_feature_comparison_df.to_csv(
    doda_dir / "rf_doda_fixed_split_feature_comparison.csv",
    index=False
)

print(f"Saved results to: {doda_dir.resolve()}")


Saved results to: C:\Users\johnm\msc_research\results\kidney_disease\doda


# 3.1 Five-Fold Stratified Cross-Validation

The fixed 80/20 split is followed by a separate **5-fold stratified cross-validation** experiment.

Feature selection is refit inside each training fold. DODA also receives all features and performs the final Top-K selection inside the training fold. Logistic Regression scaling is fitted only on the corresponding training fold.

The DODA fusion method remains **RankFusion**, matching the current BD-KDD pipeline.

In [22]:
# =============================================================================
# STEP 21: 5-FOLD CV CONFIGURATION
# =============================================================================
from sklearn.model_selection import StratifiedKFold

CV5_SPLITS = 5
CV5_RANDOM_STATE = 42
cv5 = StratifiedKFold(n_splits=CV5_SPLITS, shuffle=True, random_state=CV5_RANDOM_STATE)
cv5_results = []

print("5-fold CV configuration loaded.")
print("Splits:", CV5_SPLITS)
print("Top-K values:", TOP_K_VALUES)
print("Fusion: RankFusion")


5-fold CV configuration loaded.
Splits: 5
Top-K values: [5, 10, 15, 20]
Fusion: RankFusion


In [23]:
# =============================================================================
# STEP 22: 5-FOLD CV — RF vs DODA
# =============================================================================

# Reset results before running the CV.
# This prevents duplicate rows if the cell is executed more than once.
cv5_results = []

for fold, (train_idx, val_idx) in enumerate(
    cv5.split(X, y),
    start=1
):

    X_train_fold = X.iloc[train_idx].copy()
    X_val_fold = X.iloc[val_idx].copy()

    y_train_fold = y.iloc[train_idx].copy()
    y_val_fold = y.iloc[val_idx].copy()

    # -------------------------------------------------------------------------
    # RF feature importance is learned ONLY from the training fold.
    # -------------------------------------------------------------------------

    rf_fold_model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

    rf_fold_model.fit(X_train_fold, y_train_fold)

    rf_fold_ranked = (
        pd.Series(
            rf_fold_model.feature_importances_,
            index=X_train_fold.columns
        )
        .sort_values(ascending=False)
        .index
        .tolist()
    )

    # -------------------------------------------------------------------------
    # Evaluate each Top-K
    # -------------------------------------------------------------------------

    for top_k in TOP_K_VALUES:

        rf_features = rf_fold_ranked[:top_k]

        fold_doda = DODASelector(
            operators=[RFAdapter()],
            provider=JSONProvider(str(WEIGHTS_PATH)),
            fusion=RankFusion(),
            top_k=top_k
        )

        fold_doda.fit(
            X_train_fold,
            y_train_fold
        )

        doda_features = list(
            fold_doda.get_selected_features()
        )

        feature_sets = {
            "RF": rf_features,
            "DODA": doda_features
        }

        for method, selected_features in feature_sets.items():

            Xtr_selected = X_train_fold[selected_features]
            Xval_selected = X_val_fold[selected_features]

            for model_name, model_template in models.items():

                model = model_template.__class__(
                    **model_template.get_params()
                )

                Xtr = Xtr_selected
                Xval = Xval_selected

                if model_name == "Logistic Regression":
                    scaler = StandardScaler()
                    Xtr = scaler.fit_transform(Xtr)
                    Xval = scaler.transform(Xval)

                model.fit(Xtr, y_train_fold)

                y_pred = model.predict(Xval)
                y_prob = model.predict_proba(Xval)[:, 1]

                cv5_results.append({
                    "Fold": fold,
                    "Method": method,
                    "Top_K": top_k,
                    "Model": model_name,
                    "Selected_Features": ", ".join(selected_features),
                    "Accuracy": accuracy_score(y_val_fold, y_pred),
                    "Precision": precision_score(y_val_fold, y_pred, zero_division=0),
                    "Recall": recall_score(y_val_fold, y_pred, zero_division=0),
                    "F1": f1_score(y_val_fold, y_pred, zero_division=0),
                    "ROC_AUC": roc_auc_score(y_val_fold, y_prob)
                })

cv5_results_df = pd.DataFrame(cv5_results)

expected_rows = CV5_SPLITS * len(TOP_K_VALUES) * 2 * len(models)

print(f"Expected rows: {expected_rows}")
print(f"Actual rows:   {len(cv5_results_df)}")

assert len(cv5_results_df) == expected_rows

display(cv5_results_df.round(4))


Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x00000217F606F390>
DODA Pipeline Started
Running: RF
Operator Scores:
{'RF_1': {'Age': np.float64(0.07163829565145957), 'Bp': np.float64(0.07926881967034306), 'Sg': np.float64(0.030613050382834604), 'Al': np.float64(0.032480931220484725), 'Su': np.float64(0.03181785879926592), 'Rbc': np.float64(0.013135704570187428), 'Pc': np.float64(0.012565818029341187), 'Pcc': np.float64(0.011109063785024085), 'Ba': np.float64(0.013889361808654585), 'Bgr': np.float64(0.07524545059813406), 'Bu': np.float64(0.07689869315155583), 'Sc': np.float64(0.07654971826621065), 'Sod': np.float64(0.058706058214599226), 'Pot': np.float64(0.07267298568537175), 'Hemo': np.float64(0.07494364253794743), 'Pcv': np.float64(0.058947496042872284), 'Wbcc': np.float64(0.07590067152902041), 'Rbcc': np.float64(0.0619304804605141), 'Htn': np.float64(0.012206064677099754), 'Dm': np.float64(0.012209620944540303), 'Cad': np.float64(0

,Fold,Method,Top_K,Model,Selected_Features,Accuracy,Precision,Recall,F1,ROC_AUC
0,1,RF,5,Logistic Regression,"Bp, Bu, Sc, Wbcc, Bgr",0.5253,0.5304,0.6040,0.5648,0.5375
1,1,RF,5,Random Forest,"Bp, Bu, Sc, Wbcc, Bgr",0.4949,0.5048,0.5248,0.5146,0.5236
2,1,RF,5,XGBoost,"Bp, Bu, Sc, Wbcc, Bgr",0.5303,0.5377,0.5644,0.5507,0.5757
3,1,DODA,5,Logistic Regression,"Sc, Bu, Bp, Bgr, Hemo",0.5606,0.5686,0.5743,0.5714,0.5521
4,1,DODA,5,Random Forest,"Sc, Bu, Bp, Bgr, Hemo",0.5101,0.5179,0.5743,0.5446,0.5618
...,...,...,...,...,...,...,...,...,...,...
115,5,RF,20,Random Forest,"Bu, Sc, Bp, Pot, Hemo, Age, Wbcc, Bgr, Pcv, Rb...",0.4924,0.5043,0.5743,0.5370,0.5084
116,5,RF,20,XGBoost,"Bu, Sc, Bp, Pot, Hemo, Age, Wbcc, Bgr, Pcv, Rb...",0.4822,0.4956,0.5545,0.5234,0.5177
117,5,DODA,20,Logistic Regression,"Bu, Sc, Bp, Hemo, Pot, Al, Age, Bgr, Pcv, Sg, ...",0.5076,0.5175,0.5842,0.5488,0.5215
118,5,DODA,20,Random Forest,"Bu, Sc, Bp, Hemo, Pot, Al, Age, Bgr, Pcv, Sg, ...",0.5431,0.5478,0.6238,0.5833,0.5356


In [24]:
# =============================================================================
# STEP 23: 5-FOLD CV PERFORMANCE SUMMARY
# =============================================================================
cv5_summary_df = (
    cv5_results_df.groupby(["Method", "Top_K", "Model"])
    [["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]]
    .agg(["mean", "std"])
    .reset_index()
)
cv5_summary_df.columns = [
    "Method", "Top_K", "Model", "Accuracy_Mean", "Accuracy_STD",
    "Precision_Mean", "Precision_STD", "Recall_Mean", "Recall_STD",
    "F1_Mean", "F1_STD", "ROC_AUC_Mean", "ROC_AUC_STD"
]
display(cv5_summary_df.round(4))

cv5_dir = Path("../../../results/kidney_disease/cv")
cv5_dir.mkdir(parents=True, exist_ok=True)
cv5_results_df.to_csv(cv5_dir / "rf_doda_5fold_results.csv", index=False)
cv5_summary_df.to_csv(cv5_dir / "rf_doda_5fold_summary.csv", index=False)


,Method,Top_K,Model,Accuracy_Mean,Accuracy_STD,Precision_Mean,Precision_STD,Recall_Mean,Recall_STD,F1_Mean,F1_STD,ROC_AUC_Mean,ROC_AUC_STD
0,DODA,5,Logistic Regression,0.5375,0.0243,0.5446,0.0208,0.6076,0.0357,0.5739,0.0219,0.5531,0.0107
1,DODA,5,Random Forest,0.5071,0.0131,0.5185,0.0105,0.5425,0.0400,0.5299,0.0240,0.5062,0.0327
2,DODA,5,XGBoost,0.5263,0.0199,0.5367,0.0173,0.5622,0.0406,0.5487,0.0253,0.5278,0.0416
3,DODA,10,Logistic Regression,0.5294,0.0240,0.5374,0.0207,0.5977,0.0440,0.5655,0.0270,0.5310,0.0253
4,DODA,10,Random Forest,0.4990,0.0224,0.5125,0.0212,0.5188,0.0240,0.5152,0.0151,0.5113,0.0447
5,DODA,10,XGBoost,0.4858,0.0314,0.4985,0.0283,0.5150,0.0476,0.5064,0.0370,0.5040,0.0457
6,DODA,15,Logistic Regression,0.5344,0.0256,0.5424,0.0231,0.6017,0.0351,0.5700,0.0218,0.5328,0.0278
7,DODA,15,Random Forest,0.5293,0.0346,0.5390,0.0320,0.5760,0.0389,0.5566,0.0322,0.5204,0.0334
8,DODA,15,XGBoost,0.5061,0.0361,0.5175,0.0311,0.5484,0.0510,0.5321,0.0382,0.5247,0.0395
9,DODA,20,Logistic Regression,0.5162,0.0111,0.5271,0.0105,0.5601,0.0183,0.5429,0.0099,0.5231,0.0093


# 3.2 Robustness: 5×5 Repeated Stratified Cross-Validation

The fixed split provides a reproducible initial comparison.

For robustness, feature selection is **recomputed inside every training fold**. This prevents information from the validation fold from influencing RF feature importance or DODA.

There are:

- 5 folds
- 5 repeats
- 25 train/validation runs
- 4 Top-K budgets
- 2 feature-selection methods
- 3 downstream models

This produces 600 model-evaluation rows.


In [25]:
# =============================================================================
# STEP 24: REPEATED CV CONFIGURATION
# =============================================================================

from sklearn.model_selection import RepeatedStratifiedKFold

TOP_K_VALUES = [5, 10, 15, 20]

N_SPLITS = 5
N_REPEATS = 5
RANDOM_STATE = 42

cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE
)

print(f"Top-K values: {TOP_K_VALUES}")
print(f"CV runs: {N_SPLITS * N_REPEATS}")


Top-K values: [5, 10, 15, 20]
CV runs: 25


In [26]:
# =============================================================================
# STEP 25: 5×5 REPEATED CV — RF vs DODA
# =============================================================================

# Reset results before running the repeated CV.
cv_results = []

for run_id, (train_idx, val_idx) in enumerate(
    cv.split(X, y),
    start=1
):

    X_train_fold = X.iloc[train_idx].copy()
    X_val_fold = X.iloc[val_idx].copy()

    y_train_fold = y.iloc[train_idx].copy()
    y_val_fold = y.iloc[val_idx].copy()

    print(f"Run {run_id:02d}/{N_SPLITS * N_REPEATS}")

    # -------------------------------------------------------------------------
    # RF feature importance is learned ONLY from the training fold.
    # -------------------------------------------------------------------------

    rf_fold_model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

    rf_fold_model.fit(X_train_fold, y_train_fold)

    rf_fold_ranked = (
        pd.Series(
            rf_fold_model.feature_importances_,
            index=X_train_fold.columns
        )
        .sort_values(ascending=False)
        .index
        .tolist()
    )

    for top_k in TOP_K_VALUES:

        rf_features = rf_fold_ranked[:top_k]

        fold_doda = DODASelector(
            operators=[RFAdapter()],
            provider=JSONProvider(str(WEIGHTS_PATH)),
            fusion=RankFusion(),
            top_k=top_k
        )

        fold_doda.fit(
            X_train_fold,
            y_train_fold
        )

        doda_features = list(
            fold_doda.get_selected_features()
        )

        feature_sets = {
            "RF": rf_features,
            "DODA": doda_features
        }

        for method, selected_features in feature_sets.items():

            Xtr_selected = X_train_fold[selected_features]
            Xval_selected = X_val_fold[selected_features]

            for model_name, model_template in models.items():

                model = model_template.__class__(
                    **model_template.get_params()
                )

                Xtr = Xtr_selected
                Xval = Xval_selected

                if model_name == "Logistic Regression":
                    scaler = StandardScaler()
                    Xtr = scaler.fit_transform(Xtr)
                    Xval = scaler.transform(Xval)

                model.fit(Xtr, y_train_fold)

                y_pred = model.predict(Xval)
                y_prob = model.predict_proba(Xval)[:, 1]

                cv_results.append({
                    "Run": run_id,
                    "Method": method,
                    "Top_K": top_k,
                    "Model": model_name,
                    "Selected_Features": ", ".join(selected_features),
                    "Accuracy": accuracy_score(y_val_fold, y_pred),
                    "Precision": precision_score(y_val_fold, y_pred, zero_division=0),
                    "Recall": recall_score(y_val_fold, y_pred, zero_division=0),
                    "F1": f1_score(y_val_fold, y_pred, zero_division=0),
                    "ROC_AUC": roc_auc_score(y_val_fold, y_prob)
                })

cv_results_df = pd.DataFrame(cv_results)

expected_rows = N_SPLITS * N_REPEATS * len(TOP_K_VALUES) * 2 * len(models)

print(f"Expected repeated-CV rows: {expected_rows}")
print(f"Actual repeated-CV rows:   {len(cv_results_df)}")

assert len(cv_results_df) == expected_rows

display(cv_results_df.round(4))


Run 01/25
Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x00000217F4F8EF90>
DODA Pipeline Started
Running: RF
Operator Scores:
{'RF_1': {'Age': np.float64(0.07163829565145957), 'Bp': np.float64(0.07926881967034306), 'Sg': np.float64(0.030613050382834604), 'Al': np.float64(0.032480931220484725), 'Su': np.float64(0.03181785879926592), 'Rbc': np.float64(0.013135704570187428), 'Pc': np.float64(0.012565818029341187), 'Pcc': np.float64(0.011109063785024085), 'Ba': np.float64(0.013889361808654585), 'Bgr': np.float64(0.07524545059813406), 'Bu': np.float64(0.07689869315155583), 'Sc': np.float64(0.07654971826621065), 'Sod': np.float64(0.058706058214599226), 'Pot': np.float64(0.07267298568537175), 'Hemo': np.float64(0.07494364253794743), 'Pcv': np.float64(0.058947496042872284), 'Wbcc': np.float64(0.07590067152902041), 'Rbcc': np.float64(0.0619304804605141), 'Htn': np.float64(0.012206064677099754), 'Dm': np.float64(0.012209620944540303), 'Cad': np

,Run,Method,Top_K,Model,Selected_Features,Accuracy,Precision,Recall,F1,ROC_AUC
0,1,RF,5,Logistic Regression,"Bp, Bu, Sc, Wbcc, Bgr",0.5253,0.5304,0.6040,0.5648,0.5375
1,1,RF,5,Random Forest,"Bp, Bu, Sc, Wbcc, Bgr",0.4949,0.5048,0.5248,0.5146,0.5236
2,1,RF,5,XGBoost,"Bp, Bu, Sc, Wbcc, Bgr",0.5303,0.5377,0.5644,0.5507,0.5757
3,1,DODA,5,Logistic Regression,"Sc, Bu, Bp, Bgr, Hemo",0.5606,0.5686,0.5743,0.5714,0.5521
4,1,DODA,5,Random Forest,"Sc, Bu, Bp, Bgr, Hemo",0.5101,0.5179,0.5743,0.5446,0.5618
...,...,...,...,...,...,...,...,...,...,...
595,25,RF,20,Random Forest,"Bp, Wbcc, Sc, Hemo, Bu, Pot, Bgr, Age, Pcv, So...",0.5025,0.5138,0.5545,0.5333,0.5312
596,25,RF,20,XGBoost,"Bp, Wbcc, Sc, Hemo, Bu, Pot, Bgr, Age, Pcv, So...",0.5178,0.5273,0.5743,0.5498,0.5179
597,25,DODA,20,Logistic Regression,"Sc, Bp, Bu, Hemo, Wbcc, Al, Bgr, Pot, Pcv, Age...",0.5178,0.5259,0.6040,0.5622,0.4990
598,25,DODA,20,Random Forest,"Sc, Bp, Bu, Hemo, Wbcc, Al, Bgr, Pot, Pcv, Age...",0.5279,0.5333,0.6337,0.5792,0.5241


In [27]:
# =============================================================================
# STEP 26: CV PERFORMANCE SUMMARY
# =============================================================================

cv_summary_df = (
    cv_results_df
    .groupby(["Method", "Top_K", "Model"])
    [["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]]
    .agg(["mean", "std"])
    .reset_index()
)

cv_summary_df.columns = [
    "Method", "Top_K", "Model",
    "Accuracy_Mean", "Accuracy_STD",
    "Precision_Mean", "Precision_STD",
    "Recall_Mean", "Recall_STD",
    "F1_Mean", "F1_STD",
    "ROC_AUC_Mean", "ROC_AUC_STD"
]

display(cv_summary_df.round(4))


,Method,Top_K,Model,Accuracy_Mean,Accuracy_STD,Precision_Mean,Precision_STD,Recall_Mean,Recall_STD,F1_Mean,F1_STD,ROC_AUC_Mean,ROC_AUC_STD
0,DODA,5,Logistic Regression,0.5413,0.0290,0.5473,0.0245,0.6115,0.0471,0.5772,0.0324,0.5466,0.0336
1,DODA,5,Random Forest,0.5215,0.0386,0.5328,0.0354,0.5515,0.0447,0.5417,0.0374,0.5333,0.0419
2,DODA,5,XGBoost,0.5369,0.0334,0.5463,0.0295,0.5748,0.0505,0.5596,0.0360,0.5483,0.0431
3,DODA,10,Logistic Regression,0.5324,0.0265,0.5395,0.0229,0.6027,0.0500,0.5688,0.0322,0.5368,0.0351
4,DODA,10,Random Forest,0.5198,0.0311,0.5312,0.0290,0.5507,0.0478,0.5400,0.0339,0.5326,0.0344
5,DODA,10,XGBoost,0.5263,0.0359,0.5366,0.0336,0.5585,0.0580,0.5466,0.0416,0.5422,0.0390
6,DODA,15,Logistic Regression,0.5279,0.0323,0.5362,0.0290,0.5865,0.0551,0.5596,0.0382,0.5327,0.0366
7,DODA,15,Random Forest,0.5318,0.0315,0.5415,0.0292,0.5735,0.0444,0.5565,0.0325,0.5411,0.0326
8,DODA,15,XGBoost,0.5318,0.0340,0.5414,0.0301,0.5676,0.0609,0.5533,0.0412,0.5430,0.0304
9,DODA,20,Logistic Regression,0.5168,0.0286,0.5274,0.0262,0.5645,0.0451,0.5448,0.0311,0.5204,0.0351


In [28]:
# =============================================================================
# STEP 27: SAVE REPEATED-CV RESULTS
# =============================================================================

cv_dir = Path("../../../results/kidney_disease/cv")
cv_dir.mkdir(parents=True, exist_ok=True)

cv_results_df.to_csv(
    cv_dir / "rf_vs_doda_5x5_repeated_cv_results.csv",
    index=False
)

cv_summary_df.to_csv(
    cv_dir / "rf_vs_doda_5x5_repeated_cv_summary.csv",
    index=False
)

print(f"Saved to: {cv_dir.resolve()}")


Saved to: C:\Users\johnm\msc_research\results\kidney_disease\cv


# 4. Feature-Selection Stability, Rank/Set Change, and Statistical Comparison

Two different concepts are reported:

- **Feature-set change:** whether the Top-K membership changes.
- **Rank change:** whether feature ordering changes even when membership is retained.

Jaccard similarity evaluates set overlap, so it does not measure ordering. The fixed-split DODA rank comparison is used to inspect ordering changes.


In [29]:
# =============================================================================
# STEP 28: FEATURE-SET COMPARISON ACROSS THE 25 CV RUNS
# =============================================================================

feature_comparison_rows = []

for run_id in range(1, 26):

    for top_k in TOP_K_VALUES:

        rf_row = cv_results_df[
            (cv_results_df["Run"] == run_id) &
            (cv_results_df["Method"] == "RF") &
            (cv_results_df["Top_K"] == top_k)
        ].iloc[0]

        doda_row = cv_results_df[
            (cv_results_df["Run"] == run_id) &
            (cv_results_df["Method"] == "DODA") &
            (cv_results_df["Top_K"] == top_k)
        ].iloc[0]

        rf_set = set(
            rf_row["Selected_Features"].split(", ")
        )
        doda_set = set(
            doda_row["Selected_Features"].split(", ")
        )

        overlap = len(rf_set & doda_set)
        union = len(rf_set | doda_set)

        feature_comparison_rows.append({
            "Run": run_id,
            "Top_K": top_k,
            "Same_Feature_Set": rf_set == doda_set,
            "Overlap_Count": overlap,
            "Jaccard": overlap / union,
            "Features_Changed": top_k - overlap
        })

feature_comparison_df = pd.DataFrame(
    feature_comparison_rows
)

display(
    feature_comparison_df
    .groupby("Top_K")
    [["Same_Feature_Set", "Overlap_Count", "Jaccard", "Features_Changed"]]
    .agg(["mean", "std"])
    .round(4)
)


Same_Feature_Set         Overlap_Count         Jaccard          \
                  mean     std          mean     std    mean     std   
Top_K                                                                  
5                 0.24  0.4359          4.24  0.4359  0.7467  0.1453   
10                0.00  0.0000          8.64  0.4899  0.7636  0.0742   
15                0.04  0.2000         13.84  0.4726  0.8579  0.0537   
20                0.16  0.3742         18.68  0.8524  0.8790  0.0749   

      Features_Changed          
                  mean     std  
Top_K                           
5                 0.76  0.4359  
10                1.36  0.4899  
15                1.16  0.4726  
20                1.32  0.8524

In [30]:
# =============================================================================
# STEP 29: FEATURE-SELECTION STABILITY
# =============================================================================

from itertools import combinations

stability_rows = []

for method in ["RF", "DODA"]:

    for top_k in TOP_K_VALUES:

        subset = cv_results_df[
            (cv_results_df["Method"] == method) &
            (cv_results_df["Top_K"] == top_k)
        ]

        # One feature set per run; model does not affect selection.
        run_sets = {}

        for run_id in range(1, 26):
            row = subset[
                subset["Run"] == run_id
            ].iloc[0]

            run_sets[run_id] = set(
                row["Selected_Features"].split(", ")
            )

        pairwise_jaccards = []

        for a, b in combinations(range(1, 26), 2):

            set_a = run_sets[a]
            set_b = run_sets[b]

            pairwise_jaccards.append(
                len(set_a & set_b) / len(set_a | set_b)
            )

        stability_rows.append({
            "Method": method,
            "Top_K": top_k,
            "Mean_Jaccard": np.mean(pairwise_jaccards),
            "SD_Jaccard": np.std(pairwise_jaccards, ddof=1),
            "Min_Jaccard": np.min(pairwise_jaccards),
            "Max_Jaccard": np.max(pairwise_jaccards)
        })

stability_df = pd.DataFrame(stability_rows)

display(stability_df.round(4))


,Method,Top_K,Mean_Jaccard,SD_Jaccard,Min_Jaccard,Max_Jaccard
0,RF,5,0.6163,0.1640,0.2500,1.0
1,RF,10,0.9491,0.0818,0.8182,1.0
2,RF,15,0.9067,0.0545,0.8750,1.0
3,RF,20,0.8036,0.0617,0.6667,1.0
4,DODA,5,0.7397,0.1678,0.4286,1.0
5,DODA,10,0.8988,0.0905,0.8182,1.0
6,DODA,15,0.9048,0.0623,0.7647,1.0
7,DODA,20,0.8443,0.0630,0.6667,1.0


## 4.1 Additional Feature-Selection Stability Analyses

The 25 repeated-CV runs are used to quantify selection frequency, chance-adjusted stability, and direct Random Forest–DODA feature-set agreement.

In [31]:
# =============================================================================
# STEP 30: FEATURE SELECTION FREQUENCY
# =============================================================================
frequency_rows = []
for method in ["RF", "DODA"]:
    for top_k in TOP_K_VALUES:
        subset = cv_results_df[(cv_results_df["Method"] == method) & (cv_results_df["Top_K"] == top_k)]
        counts = {feature: 0 for feature in X.columns}
        for run_id in sorted(subset["Run"].unique()):
            row = subset[subset["Run"] == run_id].iloc[0]
            for feature in row["Selected_Features"].split(", "):
                counts[feature] += 1
        for feature, count in counts.items():
            frequency_rows.append({
                "Method": method, "Top_K": top_k, "Feature": feature,
                "Selection_Count": count,
                "Selection_Frequency": count / (N_SPLITS * N_REPEATS)
            })
feature_frequency_df = pd.DataFrame(frequency_rows)

display(
    feature_frequency_df.sort_values(
        ["Method", "Top_K", "Selection_Frequency", "Feature"],
        ascending=[True, True, False, True]
    ).groupby(["Method", "Top_K"], group_keys=False).head(10).round(4)
)


,Method,Top_K,Feature,Selection_Count,Selection_Frequency
97,DODA,5,Bp,25,1.00
106,DODA,5,Bu,25,1.00
107,DODA,5,Sc,25,1.00
110,DODA,5,Hemo,21,0.84
109,DODA,5,Pot,16,0.64
...,...,...,...,...,...
86,RF,20,Hemo,25,1.00
87,RF,20,Pcv,25,1.00
85,RF,20,Pot,25,1.00
89,RF,20,Rbcc,25,1.00


In [32]:
# =============================================================================
# STEP 31: KUNCHEVA INDEX
# =============================================================================
from itertools import combinations
N_FEATURES = X.shape[1]
kuncheva_rows = []

for method in ["RF", "DODA"]:
    for top_k in TOP_K_VALUES:
        subset = cv_results_df[(cv_results_df["Method"] == method) & (cv_results_df["Top_K"] == top_k)]
        run_sets = {}
        for run_id in sorted(subset["Run"].unique()):
            row = subset[subset["Run"] == run_id].iloc[0]
            run_sets[run_id] = set(row["Selected_Features"].split(", "))

        for run_a, run_b in combinations(sorted(run_sets), 2):
            intersection = len(run_sets[run_a] & run_sets[run_b])
            denominator = top_k * (N_FEATURES - top_k)
            value = np.nan if denominator == 0 else ((intersection * N_FEATURES) - (top_k ** 2)) / denominator
            kuncheva_rows.append({
                "Method": method, "Top_K": top_k, "Run_A": run_a, "Run_B": run_b,
                "Intersection": intersection, "Kuncheva_Index": value
            })

kuncheva_df = pd.DataFrame(kuncheva_rows)
kuncheva_summary_df = (
    kuncheva_df.groupby(["Method", "Top_K"])["Kuncheva_Index"]
    .agg(Mean_Kuncheva="mean", Std_Kuncheva="std", Min_Kuncheva="min", Max_Kuncheva="max")
    .reset_index()
)
display(kuncheva_summary_df.round(4))


,Method,Top_K,Mean_Kuncheva,Std_Kuncheva,Min_Kuncheva,Max_Kuncheva
0,DODA,5,0.7979,0.1371,0.4947,1.0
1,DODA,10,0.9046,0.0853,0.8286,1.0
2,DODA,15,0.8637,0.0906,0.6444,1.0
3,DODA,20,0.4860,0.2244,-0.2000,1.0
4,RF,5,0.6842,0.1589,0.2421,1.0
5,RF,10,0.9520,0.0771,0.8286,1.0
6,RF,15,0.8673,0.0774,0.8222,1.0
7,RF,20,0.3390,0.2266,-0.2000,1.0


In [33]:
# =============================================================================
# STEP 32: RF–DODA FEATURE-SET AGREEMENT SUMMARY
# =============================================================================
agreement_summary_df = (
    feature_comparison_df.groupby("Top_K")
    .agg(
        Same_Feature_Set_Rate=("Same_Feature_Set", "mean"),
        Mean_Overlap=("Overlap_Count", "mean"),
        SD_Overlap=("Overlap_Count", "std"),
        Mean_Jaccard=("Jaccard", "mean"),
        SD_Jaccard=("Jaccard", "std"),
        Mean_Features_Changed=("Features_Changed", "mean"),
        SD_Features_Changed=("Features_Changed", "std")
    ).reset_index()
)
display(agreement_summary_df.round(4))


,Top_K,Same_Feature_Set_Rate,Mean_Overlap,SD_Overlap,Mean_Jaccard,SD_Jaccard,Mean_Features_Changed,SD_Features_Changed
0,5,0.24,4.24,0.4359,0.7467,0.1453,0.76,0.4359
1,10,0.00,8.64,0.4899,0.7636,0.0742,1.36,0.4899
2,15,0.04,13.84,0.4726,0.8579,0.0537,1.16,0.4726
3,20,0.16,18.68,0.8524,0.8790,0.0749,1.32,0.8524


In [34]:
# =============================================================================
# STEP 33: PAIRED STATISTICAL COMPARISON OF PREDICTIVE PERFORMANCE
# =============================================================================

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]

comparison_rows = []

for top_k in TOP_K_VALUES:

    for model_name in models.keys():

        rf = cv_results_df[
            (cv_results_df["Method"] == "RF") &
            (cv_results_df["Top_K"] == top_k) &
            (cv_results_df["Model"] == model_name)
        ].sort_values("Run")

        doda = cv_results_df[
            (cv_results_df["Method"] == "DODA") &
            (cv_results_df["Top_K"] == top_k) &
            (cv_results_df["Model"] == model_name)
        ].sort_values("Run")

        assert len(rf) == 25
        assert len(doda) == 25
        assert (rf["Run"].values == doda["Run"].values).all()

        for metric in metrics:

            rf_values = rf[metric].to_numpy()
            doda_values = doda[metric].to_numpy()

            differences = doda_values - rf_values

            mean_difference = np.mean(differences)
            sd_difference = np.std(differences, ddof=1)

            if np.allclose(differences, 0):
                statistic = 0.0
                p_value = 1.0
            else:
                statistic, p_value = wilcoxon(
                    doda_values,
                    rf_values,
                    alternative="two-sided"
                )

            paired_d = (
                mean_difference / sd_difference
                if sd_difference > 0 else 0.0
            )

            comparison_rows.append({
                "Top_K": top_k,
                "Model": model_name,
                "Metric": metric,
                "RF_Mean": np.mean(rf_values),
                "DODA_Mean": np.mean(doda_values),
                "Mean_Difference_DODA_minus_RF": mean_difference,
                "Paired_SD": sd_difference,
                "Paired_Cohens_d": paired_d,
                "Wilcoxon_Statistic": statistic,
                "P_Value": p_value
            })

performance_comparison_df = pd.DataFrame(comparison_rows)

reject, adjusted_p, _, _ = multipletests(
    performance_comparison_df["P_Value"],
    alpha=0.05,
    method="holm"
)

performance_comparison_df["Holm_Adjusted_P"] = adjusted_p
performance_comparison_df["Significant"] = reject

display(performance_comparison_df.round(4))


,Top_K,Model,Metric,RF_Mean,DODA_Mean,Mean_Difference_DODA_minus_RF,Paired_SD,Paired_Cohens_d,Wilcoxon_Statistic,P_Value,Holm_Adjusted_P,Significant
0,5,Logistic Regression,Accuracy,0.5348,0.5413,0.0065,0.0284,0.2283,60.5,0.4484,1.0000,False
1,5,Logistic Regression,Precision,0.5414,0.5473,0.0058,0.0248,0.2357,56.0,0.3318,1.0000,False
2,5,Logistic Regression,Recall,0.6131,0.6115,-0.0016,0.0239,-0.0671,30.5,0.5044,1.0000,False
3,5,Logistic Regression,F1,0.5746,0.5772,0.0025,0.0231,0.1092,76.0,0.9811,1.0000,False
4,5,Logistic Regression,ROC_AUC,0.5357,0.5466,0.0109,0.0304,0.3594,36.0,0.0176,0.9671,False
5,5,Random Forest,Accuracy,0.5071,0.5215,0.0144,0.0408,0.3534,102.5,0.1064,1.0000,False
6,5,Random Forest,Precision,0.5192,0.5328,0.0136,0.0382,0.3569,102.0,0.1073,1.0000,False
7,5,Random Forest,Recall,0.5369,0.5515,0.0146,0.0457,0.3198,93.5,0.1062,1.0000,False
8,5,Random Forest,F1,0.5273,0.5417,0.0143,0.0380,0.3765,97.0,0.0802,1.0000,False
9,5,Random Forest,ROC_AUC,0.5069,0.5333,0.0264,0.0364,0.7247,39.0,0.0004,0.0258,True


In [35]:
# =============================================================================
# STEP 34: SAVE STATISTICAL RESULTS
# =============================================================================

stats_dir = Path("../../../results/kidney_disease/statistics")
stats_dir.mkdir(parents=True, exist_ok=True)

feature_comparison_df.to_csv(
    stats_dir / "feature_set_comparison_5x5.csv",
    index=False
)

stability_df.to_csv(
    stats_dir / "feature_selection_stability_5x5.csv",
    index=False
)

performance_comparison_df.to_csv(
    stats_dir / "rf_vs_doda_predictive_statistics_5x5.csv",
    index=False
)

print(f"Saved statistical outputs to: {stats_dir.resolve()}")

feature_frequency_df.to_csv(
    stats_dir / "feature_selection_frequency_5x5.csv",
    index=False
)

kuncheva_df.to_csv(
    stats_dir / "kuncheva_pairwise_5x5.csv",
    index=False
)

kuncheva_summary_df.to_csv(
    stats_dir / "kuncheva_summary_5x5.csv",
    index=False
)

agreement_summary_df.to_csv(
    stats_dir / "rf_doda_agreement_summary_5x5.csv",
    index=False
)


Saved statistical outputs to: C:\Users\johnm\msc_research\results\kidney_disease\statistics


In [36]:
# =============================================================================
# STEP 36: FINAL SANITY CHECKS
# =============================================================================

print("=" * 70)
print("FINAL EXPERIMENT SANITY CHECK")
print("=" * 70)

print(f"Total predictors: {X.shape[1]}")
print(f"Top-K values: {TOP_K_VALUES}")
print(f"Missing values: {int(X.isna().sum().sum())}")
print(f"5-fold CV result rows: {len(cv5_results_df)}")
print(f"Repeated CV result rows: {len(cv_results_df)}")
print(f"Expected 5-fold rows: {5 * len(TOP_K_VALUES) * 2 * len(models)}")
print(f"Expected repeated CV rows: {25 * len(TOP_K_VALUES) * 2 * len(models)}")

assert X.shape[1] == 24
assert X.isna().sum().sum() == 0
assert TOP_K_VALUES == [5, 10, 15, 20]
assert len(cv5_results_df) == 5 * len(TOP_K_VALUES) * 2 * len(models)
assert len(cv_results_df) == 25 * len(TOP_K_VALUES) * 2 * len(models)
assert len(feature_frequency_df) == 2 * len(TOP_K_VALUES) * X.shape[1]
assert len(kuncheva_summary_df) == 2 * len(TOP_K_VALUES)
assert len(agreement_summary_df) == len(TOP_K_VALUES)

for top_k in TOP_K_VALUES:
    assert len(rf_results[top_k]["features"]) == top_k
    assert len(rank_doda_results[top_k]["features"]) == top_k

print("All sanity checks passed.")


FINAL EXPERIMENT SANITY CHECK
Total predictors: 24
Top-K values: [5, 10, 15, 20]
Missing values: 0
5-fold CV result rows: 120
Repeated CV result rows: 600
Expected 5-fold rows: 120
Expected repeated CV rows: 600
All sanity checks passed.
